### Inital look and saving cusips to CSV

In [1]:
import duckdb
import pandas as pd
db_path = r"C:\Users\agand\Desktop\Personal_Coding\Finance_Projects\MultiFactorModel_LFQA\multi-factor-models\venv\crsp\crsp_dataset.duckdb"
parquet_file = r"C:\Users\agand\Desktop\Personal_Coding\Finance_Projects\MultiFactorModel_LFQA\multi-factor-models\venv\crsp\CRSPdF20022025.parquet"

conn = duckdb.connect(db_path)

conn.execute(f"""
CREATE TABLE IF NOT EXISTS crsp_table AS
SELECT * FROM read_parquet('{parquet_file}')
""")

df_preview = conn.execute("SELECT * FROM crsp_table LIMIT 5").fetchdf()
df_preview


,cusip,date,permno,permco,cfacpr,cfacshr,shrout,prc,ret,retx,vol,ask,askhi,bid,bidlo
0,36720410,2002-01-02,10001,7953,1.5,1.5,2566.0,11.10,-0.030568,-0.030568,6603.0,11.35,11.49,11.19,11.1000
1,05978R10,2002-01-02,10002,7954,1.0,1.0,8530.0,9.60,-0.029323,-0.029323,55000.0,9.89,9.60,9.27,9.6000
2,23326910,2002-01-02,10012,7969,1.0,1.0,20924.0,3.25,0.041667,0.041667,50975.0,3.25,3.29,3.24,3.0600
3,44950710,2002-01-02,10019,7971,1.0,1.0,8282.0,1.05,-0.160000,-0.160000,53250.0,1.05,1.30,1.03,1.0497
4,00103110,2002-01-02,10025,7975,1.0,1.0,7794.0,23.80,-0.006678,-0.006678,7030.0,23.90,24.00,23.80,23.4000


In [3]:
# Get all unique CUSIPs
cusip_df = conn.execute("""
SELECT DISTINCT cusip 
FROM crsp_table
ORDER BY cusip
""").fetchdf()

print(f"Number of unique CUSIPs: {len(cusip_df)}")
cusip_df



Number of unique CUSIPs: 20553


,cusip
0,00030710
1,00032Q10
2,00036020
3,00036110
4,00037520
...,...
20548,Y9233510
20549,Y9369110
20550,Y9384M10
20551,Y9390M10


In [2]:
#cusip_df.to_csv("cusips.csv")
conn.close()

### Beginning to load factors and work on data

In [3]:
import pandas as pd
import duckdb

#### Cleaning and alinging the data

In [11]:
import duckdb

db_path = r"C:\Users\agand\Desktop\Personal_Coding\Finance_Projects\MultiFactorModel_LFQA\multi-factor-models\venv\crsp\crsp_dataset.duckdb"
parquet_file = r"C:\Users\agand\Desktop\Personal_Coding\Finance_Projects\MultiFactorModel_LFQA\multi-factor-models\venv\crsp\CRSPdF20022025.parquet"

con = duckdb.connect(db_path)

con.execute(f"""
CREATE TABLE IF NOT EXISTS crsp_table AS
SELECT * FROM read_parquet('{parquet_file}');
""")

In [12]:
con.execute("""
DROP TABLE IF EXISTS crsp_clean;
CREATE TABLE crsp_clean AS
SELECT
  cusip,
  CAST(date AS DATE) AS date,
  permno,
  permco,
  cfacpr,
  cfacshr,
  CASE WHEN shrout IS NULL OR shrout <= 0 THEN NULL ELSE shrout END AS shrout,
  CASE WHEN prc    IS NULL OR prc    <= 0 THEN NULL ELSE prc    END AS prc,
  ret,
  retx,
  CASE WHEN vol    IS NULL OR vol    <= 0 THEN NULL ELSE vol    END AS vol,
  CASE WHEN ask    IS NULL OR ask    <= 0 THEN NULL ELSE ask    END AS ask,
  askhi,
  CASE WHEN bid    IS NULL OR bid    <= 0 THEN NULL ELSE bid    END AS bid,
  bidlo
FROM crsp_table
WHERE cusip IS NOT NULL AND date IS NOT NULL
ORDER BY cusip, date;
""")

In [13]:
summary = con.execute("""
SELECT 
  COUNT(*) AS total_rows,
  COUNT(DISTINCT cusip) AS unique_cusips,
  MIN(date) AS min_date,
  MAX(date) AS max_date,
  SUM(prc    IS NULL)::BIGINT AS null_prc,
  SUM(shrout IS NULL)::BIGINT AS null_shrout,
  SUM(vol    IS NULL)::BIGINT AS null_vol,
  SUM(ask    IS NULL)::BIGINT AS null_ask,
  SUM(bid    IS NULL)::BIGINT AS null_bid
FROM crsp_clean;
""").fetchdf()
print("Summary: ")
print(summary)

dupes = con.execute("""
SELECT cusip, date, COUNT(*) AS n
FROM crsp_clean
GROUP BY 1,2
HAVING COUNT(*) > 1
LIMIT 25;
""").fetchdf()
print("Sample duplicates per (cusip, date):")
print(dupes)

Summary: 
   total_rows  unique_cusips   min_date   max_date  null_prc  null_shrout  \
0    40802455          20553 2002-01-02 2023-12-29   1758743            1   

   null_vol  null_ask  null_bid  
0   1566793    565855    565815  
Sample duplicates per (cusip, date):
Empty DataFrame
Columns: [cusip, date, n]
Index: []


In [14]:
con.execute("""
DROP TABLE IF EXISTS crsp_gaps;
CREATE TABLE crsp_gaps AS
WITH seq AS (
  SELECT
    cusip,
    date,
    LAG(date) OVER (PARTITION BY cusip ORDER BY date) AS prev_date
  FROM crsp_clean
)
SELECT
  cusip,
  date,
  prev_date,
  CASE
    WHEN prev_date IS NULL THEN NULL
    ELSE DATE_DIFF('day', prev_date, date)
  END AS gap_days
FROM seq
WHERE prev_date IS NOT NULL;
""")


In [15]:
con.execute("""
DROP TABLE IF EXISTS crsp_continuity;
CREATE TABLE crsp_continuity AS
SELECT
  c.cusip,
  MIN(c.date) AS start_date,
  MAX(c.date) AS end_date,
  COUNT(*) AS obs,
  COUNT(DISTINCT c.date) AS distinct_dates,
  SUM(CASE WHEN g.gap_days IS NOT NULL THEN 1 ELSE 0 END) AS transitions,
  SUM(CASE WHEN g.gap_days > 7 THEN 1 ELSE 0 END) AS gaps_gt_7d,
  MAX(COALESCE(g.gap_days,0)) AS max_gap_days,
  AVG(CASE WHEN g.gap_days IS NOT NULL THEN g.gap_days END) AS avg_gap_days
FROM crsp_clean c
LEFT JOIN crsp_gaps g
  ON c.cusip = g.cusip AND c.date = g.date
GROUP BY 1;
""")

In [16]:
continuity_flag = con.execute("""
SELECT *
FROM crsp_continuity
ORDER BY gaps_gt_7d DESC, max_gap_days DESC
LIMIT 50;
""").fetchdf()
print("Worst continuity cases:")
print(continuity_flag)

Worst continuity cases:
       cusip start_date   end_date   obs  distinct_dates  transitions  \
0   78470V10 2011-07-27 2020-01-14  2131            2131       2130.0   
1   22087310 2002-01-02 2009-09-14  1939            1939       1938.0   
2   44894750 2002-01-02 2023-12-29  5537            5537       5536.0   
3   46138E53 2007-10-11 2023-12-29  4083            4083       4082.0   
4   16938G10 2007-10-05 2017-05-16  2420            2420       2419.0   
5   49918380 2003-12-18 2012-07-16  2159            2159       2158.0   
6   59132A10 2010-04-09 2013-04-12   758             758        757.0   
7   29384810 2002-01-02 2009-07-28  1906            1906       1905.0   
8   36146210 2002-01-02 2008-04-30  1593            1593       1592.0   
9   24610V10 2002-01-02 2022-02-11  5065            5065       5064.0   
10  78464A70 2002-01-02 2023-12-29  5537            5537       5536.0   
11  45661Q10 2005-06-21 2010-11-05  1356            1356       1355.0   
12  73935X55 2006-09-20 200

In [17]:
con.execute("""
CREATE OR REPLACE VIEW v_crsp_base AS
SELECT
  cusip, date, permno, permco,
  prc, shrout, vol, ask, bid, askhi, bidlo, ret, retx,
  /* pre-compute useful primitives; results will be NULL if inputs are NULL */
  (prc * shrout * 1000) AS market_cap,
  CASE WHEN shrout IS NULL OR vol IS NULL THEN NULL ELSE vol / shrout END AS turnover,
  CASE 
    WHEN ask IS NULL OR bid IS NULL OR (ask + bid) = 0 THEN NULL
    ELSE (ask - bid) / ((ask + bid) / 2.0)
  END AS bid_ask_spread
FROM crsp_clean;
""")


In [20]:
print("Views and tables ready: crsp_table, crsp_clean, crsp_gaps, crsp_continuity, v_crsp_base")
con.execute("SHOW TABLES;").fetchdf()


Views and tables ready: crsp_table, crsp_clean, crsp_gaps, crsp_continuity, v_crsp_base


,name
0,crsp_clean
1,crsp_continuity
2,crsp_gaps
3,crsp_table
4,v_crsp_base


#### Making Base Variables

In [21]:
con.execute("""
DROP TABLE IF EXISTS factors_daily;
CREATE TABLE factors_daily AS
SELECT
  c.cusip,
  c.date,
  c.permno,
  c.permco,
  c.prc,
  c.shrout,
  c.vol,
  c.ask,
  c.bid,
  c.askhi,
  c.bidlo,
  c.ret,
  c.retx,
  /* Market cap: prc * shrout * 1000 (shrout in thousands) */
  CASE WHEN c.prc IS NULL OR c.shrout IS NULL
       THEN NULL
       ELSE c.prc * c.shrout * 1000
  END AS market_cap,
  /* Turnover: vol / shrout */
  CASE WHEN c.vol IS NULL OR c.shrout IS NULL OR c.shrout = 0
       THEN NULL
       ELSE c.vol / c.shrout
  END AS turnover,
  /* Bid-ask spread: (ask - bid) / ((ask + bid)/2) */
  CASE 
    WHEN c.ask IS NULL OR c.bid IS NULL OR (c.ask + c.bid) = 0
    THEN NULL
    ELSE (c.ask - c.bid) / ((c.ask + c.bid) / 2.0)
  END AS bid_ask_spread
FROM crsp_clean c
ORDER BY c.cusip, c.date;
""")


Monthly aggregation
- market_cap_month_end: value at month-end (arg_max by date)
- turnover_month_avg: simple monthly average of daily turnover
- spread_month_avg: simple monthly average of daily spread

In [22]:

con.execute("""
DROP TABLE IF EXISTS factors_monthly;
CREATE TABLE factors_monthly AS
WITH tagged AS (
  SELECT
    cusip,
    date,
    date_trunc('month', date) AS month,
    market_cap,
    turnover,
    bid_ask_spread
  FROM factors_daily
)
SELECT
  cusip,
  month AS month_start,
  /* month_end for reference */
  (DATE_TRUNC('month', month) + INTERVAL 1 MONTH - INTERVAL 1 DAY)::DATE AS month_end,
  /* market cap at month-end (last available day in month) */
  ARG_MAX(market_cap, date) AS market_cap_month_end,
  /* monthly averages (NULLs ignored) */
  AVG(turnover) AS turnover_month_avg,
  AVG(bid_ask_spread) AS spread_month_avg,
  COUNT(*) AS trading_days_in_month
FROM tagged
GROUP BY cusip, month
ORDER BY cusip, month;
""")

Quick sanity pulls

In [23]:
print(con.execute("""
SELECT COUNT(*) AS daily_rows, COUNT(DISTINCT cusip) AS cusips
FROM factors_daily;
""").fetchdf())

print(con.execute("""
SELECT COUNT(*) AS monthly_rows, MIN(month_start) AS min_m, MAX(month_start) AS max_m
FROM factors_monthly;
""").fetchdf())

print("Built: factors_daily, factors_monthly.")


   daily_rows  cusips
0    40802455   20553
   monthly_rows      min_m      max_m
0       1956558 2002-01-01 2023-12-01
Built: factors_daily, factors_monthly.


In [24]:
con.execute("SHOW TABLES;").fetchdf()

,name
0,crsp_clean
1,crsp_continuity
2,crsp_gaps
3,crsp_table
4,factors_daily
5,factors_monthly
6,v_crsp_base


#### Computing rolling factors